# Univ-Us F9 — Qwen3-8B QLoRA (캐글 T4×2 · fp16)

`kaggle/qwen3-8b` 브랜치. 설정값은 저장소의 `config.env` 에서 읽는다 — 이 노트북에는 모델 이름을 적지 않는다.

**실행 전 확인** (setup_kaggle.md): Accelerator **GPU T4 x2**(한 장만 씀 — 1번 셀 `CUDA_VISIBLE_DEVICES=0`) · Internet **On** · Secrets 에 **HF_TOKEN**(write) ·
`config.env` 의 `HF_PUSH_REPO` `HF_DATA_REPO` 를 본인 HF id 로. 세션이 끝나면 `/kaggle/working` 은 사라진다 — 결과는 HF Hub 에만 남는다.

In [ ]:
# 1) 코드 가져오기 — REPO_URL 이 있으면 git clone, 없으면 Kaggle Dataset(univus-train-code)에서 config.env 를 재귀로 찾아 복사
import os, shutil, subprocess, pathlib
# T4 한 장만 쓴다 — 8B nf4 는 한 장에 들어가고, device_map="auto" 로 두 장에 나누면 PCIe 를 오가느라 더 느리다.
# subprocess(train/eval) 가 상속하도록 어떤 서브프로세스보다 먼저 둔다.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO_URL = ""                     # 예: https://github.com/<org>/Univ-Us_Train.git  (비우면 아래 Dataset 경로 사용)
BRANCH   = "kaggle/qwen3-8b"
CODE_DS  = pathlib.Path("/kaggle/input/univus-train-code")
WORK     = pathlib.Path("/kaggle/working/Univ-Us_Train")
if WORK.exists():
    shutil.rmtree(WORK)
if REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(WORK)], check=True)
else:
    assert CODE_DS.exists(), "REPO_URL 이 비어 있고 Kaggle Dataset(univus-train-code)도 없습니다 — setup_kaggle.md 3장 (kernel-metadata.json dataset_sources)"
    hits = sorted(CODE_DS.rglob("config.env"))          # zip 이 한 단계 더 안에 풀려 있어도 찾는다
    assert hits, f"{CODE_DS} 아래에 config.env 가 없습니다 — 브랜치 체크아웃 폴더를 통째로 올렸는지 확인"
    shutil.copytree(hits[0].parent, WORK)
    print("코드 출처:", hits[0].parent)
os.chdir(WORK)
print(sorted(p.name for p in WORK.iterdir()))

In [ ]:
# 2) config.env 읽기 (bash KEY=value → dict). RUN_NAME 참조(${RUN_NAME}) 만 간단히 치환한다
import re
cfg = {}
for line in pathlib.Path("config.env").read_text(encoding="utf-8").splitlines():
    line = line.split("#", 1)[0].strip()
    if "=" in line:
        k, v = line.split("=", 1); cfg[k.strip()] = v.strip()
for k, v in cfg.items():
    cfg[k] = re.sub(r"\$\{(\w+)\}", lambda m: cfg.get(m.group(1), ""), v)
assert cfg.get("MODEL_ID"), "config.env 의 MODEL_ID 가 비어 있습니다"
assert cfg.get("COMPUTE_DTYPE") == "fp16", "캐글 T4/P100 은 bf16 이 없습니다 — config.env COMPUTE_DTYPE=fp16"
for k in ("HF_PUSH_REPO", "HF_DATA_REPO"):
    assert cfg.get(k) and not cfg[k].startswith("your-hf-id"), f"config.env 의 {k} 를 본인 HF id 로 바꾸세요"
cfg

In [ ]:
# 3) 패키지 — 캐글 이미지의 torch(CUDA) 는 그대로 두고 나머지만 올린다
%pip install -q -U -r requirements.txt
import torch, importlib.metadata as md
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "bf16", torch.cuda.is_bf16_supported())
for n in ("transformers", "peft", "trl", "datasets", "bitsandbytes"):
    print(f"  {n:14} {md.version(n)}")
assert torch.cuda.is_available(), "GPU 가 켜져 있지 않습니다 — Settings → Accelerator"
if torch.cuda.is_bf16_supported():
    print("이 GPU 는 bf16 을 지원합니다(A100/L4?) — config.env 를 bf16 으로 바꿔도 됩니다")

In [ ]:
# 4) HF 로그인 — Secrets 의 HF_TOKEN (없으면 환경변수 HF_TOKEN)
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN", "")
assert token, "HF_TOKEN 이 없습니다 — Add-ons → Secrets 에 추가 (write 권한)"
login(token=token, add_to_git_credential=False)
os.environ["HF_TOKEN"] = token
print("HF 로그인 OK")

In [ ]:
# 5) 데이터 — HF private dataset 에서 받는다. 없으면 합성 생성(⚠️ 그러면 팀 평가셋과 달라져 비교가 안 된다)
from huggingface_hub import snapshot_download
DATA_DIR = pathlib.Path(cfg.get("DATA_DIR", "data/f9"))
try:
    snapshot_download(cfg["HF_DATA_REPO"], repo_type="dataset", local_dir=str(DATA_DIR))
    print("데이터 받음 →", DATA_DIR, sorted(p.name for p in DATA_DIR.iterdir()))
except Exception as e:
    print("!!! HF 데이터셋을 못 받음:", str(e)[:200])
    print("!!! 합성 데이터를 새로 만듭니다 — 팀 평가셋과 다르므로 compare.md 를 다른 실험과 비교하지 마세요")
    subprocess.run(["python", "src/prepare_f9.py", "--out", str(DATA_DIR), "--seed", cfg.get("SEED", "42"),
                    "--n", cfg.get("N_ROWS", "1500")], check=True)

In [ ]:
# 6) 학습 — 끝나면 어댑터를 HF_PUSH_REPO 에 올린다 (세션이 끝나면 /kaggle/working 은 사라진다)
OUT = cfg.get("OUTPUT_DIR") or f"outputs/{cfg.get('RUN_NAME', 'run1')}"
cmd = ["python", "src/train.py", "--model_id", cfg["MODEL_ID"], "--data_dir", str(DATA_DIR), "--output_dir", OUT,
       "--load_4bit", cfg.get("LOAD_4BIT", "1"), "--compute_dtype", cfg.get("COMPUTE_DTYPE", "fp16"),
       "--max_seq", cfg.get("MAX_SEQ", "2560"), "--batch", cfg.get("BATCH", "1"), "--grad_accum", cfg.get("GRAD_ACCUM", "8"),
       "--lora_r", cfg.get("LORA_R", "16"), "--epochs", cfg.get("EPOCHS", "3"), "--lr", cfg.get("LR", "2e-4"),
       "--save_steps", cfg.get("SAVE_STEPS", "50"), "--seed", cfg.get("SEED", "42"),
       "--push_to_hub", cfg["HF_PUSH_REPO"]]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# 7) 평가 — 학습 전 / 학습 후 를 같은 eval.jsonl 로. 결과 md 도 HF repo 에 올린다
from huggingface_hub import HfApi
ev = str(DATA_DIR / "eval.jsonl")
subprocess.run(["python", "src/eval.py", "--model_id", cfg["MODEL_ID"], "--data", ev,
                "--out", f"{OUT}/eval_base.md", "--label", "학습 전 Instruct", "--compute_dtype", "fp16"], check=True)
subprocess.run(["python", "src/eval.py", "--model_id", cfg["MODEL_ID"], "--adapter", f"{OUT}/adapter", "--data", ev,
                "--out", f"{OUT}/eval_adapter.md", "--label", "학습 후", "--compute_dtype", "fp16"], check=True)
rows = "".join(p.read_text(encoding="utf-8") for p in sorted(pathlib.Path(OUT).glob("summary_row_*.md")))
compare = ("# F9 비교표 — " + cfg.get("RUN_NAME", "") + " (" + cfg["MODEL_ID"] + ", 캐글 T4 fp16)\n\n"
           "| 구분 | 턴 n | 도구 선택 | 인자 일치 | 필드 정확도 | 거절 정확도 | 텍스트 턴 정확도 | 턴 정확도 | 평균 지연(s) |\n"
           "| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |\n" + rows)
pathlib.Path(OUT, "compare.md").write_text(compare, encoding="utf-8")
print(compare)
api = HfApi()
for name in ("eval_base.md", "eval_base.json", "eval_adapter.md", "eval_adapter.json", "compare.md", "run.json"):
    p = pathlib.Path(OUT, name)
    if p.exists():
        api.upload_file(path_or_fileobj=str(p), path_in_repo=name, repo_id=cfg["HF_PUSH_REPO"], repo_type="model")
print("HF 업로드 완료 →", cfg["HF_PUSH_REPO"])

## 끝난 뒤

HF 의 `HF_PUSH_REPO` 에 `adapter_config.json` `adapter_model.safetensors` `run.json` `compare.md` 가 있어야 한다. **없으면 결과가 없는 것이다.**
서버에서 쓰려면 `hf download <HF_PUSH_REPO> --local-dir ~/univus-train-out/outputs/<RUN_NAME>/adapter` (huggingface_hub 2.x 의 CLI 는 `hf`).
중간에 끊겼으면 `hf download <HF_PUSH_REPO> --include "last-checkpoint/*" --local-dir outputs/<RUN_NAME>` 로 받아 `checkpoint-N` 으로 두고 다음 세션에서 `--resume`.